# YOLOv8 fracture model: mAP on GRAZPEDWRI-DX

Evaluates the weights published by Ju & Cai (`best.pt`: YOLOv8s, 640 px, SGD) on their own
validation and test splits, and compares with the paper.

Run on Google Colab with a GPU (Runtime > Change runtime type > T4 GPU). The images are about 16 GB,
the download takes 10-20 minutes.

In [ ]:
!pip -q install ultralytics
!git clone -q --depth 1 https://github.com/RuiyangJu/Bone_Fracture_Detection_YOLOv8 ju

## Data
GRAZPEDWRI-DX from figshare (CC BY 4.0). Only the validation and test images of the authors' split
(`csv/valid_data.csv`, `csv/test_data.csv`) are kept.

In [ ]:
import csv, subprocess, zipfile
from pathlib import Path

root = Path("grazpedwri")
parts = [34268828, 34268849, 34268864, 34268891]
wanted = {}
for split in ("valid", "test"):
    with open(f"ju/csv/{split}_data.csv", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            wanted[row["filestem"]] = split
for kind in ("images", "labels"):
    for split in ("valid", "test"):
        (root / kind / split).mkdir(parents=True, exist_ok=True)

subprocess.run(["wget", "-q", "-O", "labels.zip", "https://ndownloader.figshare.com/files/34268819"], check=True)
with zipfile.ZipFile("labels.zip") as z:
    for stem, split in wanted.items():
        (root / "labels" / split / f"{stem}.txt").write_bytes(z.read(f"yolov5/labels/{stem}.txt"))

for fid in parts:
    subprocess.run(["wget", "-q", "-O", "part.zip", f"https://ndownloader.figshare.com/files/{fid}"], check=True)
    with zipfile.ZipFile("part.zip") as z:
        for entry in z.namelist():
            stem = Path(entry).stem
            if entry.endswith(".png") and stem in wanted:
                (root / "images" / wanted[stem] / f"{stem}.png").write_bytes(z.read(entry))
    Path("part.zip").unlink()
print({s: len(list((root / "images" / s).glob("*.png"))) for s in ("valid", "test")})

In [ ]:
Path("grazpedwri.yaml").write_text(f"""path: {root.resolve()}
train: images/valid
val: images/valid
test: images/test
names:
  0: boneanomaly
  1: bonelesion
  2: foreignbody
  3: fracture
  4: metal
  5: periostealreaction
  6: pronatorsign
  7: softtissue
  8: text
""")

## Model

In [ ]:
import hashlib, urllib.request
url = "https://github.com/RuiyangJu/Bone_Fracture_Detection_YOLOv8/releases/download/Trained_model/best.pt"
urllib.request.urlretrieve(url, "yolov8-fracture.pt")
assert hashlib.sha256(open("yolov8-fracture.pt", "rb").read()).hexdigest() == \
    "301abc1774c28dd7c5adbcf1e8a79ed6771273615238b62dd179b622292b3a81"

## Evaluation
Same settings as the paper (`conf=0.001`, `iou=0.6`, 640 px).

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8-fracture.pt")
results = {}
for split in ("val", "test"):
    m = model.val(data="grazpedwri.yaml", split=split, imgsz=640, batch=16, conf=0.001, iou=0.6, plots=False)
    results[split] = (m.box.map50, m.box.map, m.box.ap50[3], m.box.ap[3])
for split, (map50, map5095, fr50, fr5095) in results.items():
    print(f"{split:5}  mAP50 {map50:.3f}  mAP50-95 {map5095:.3f}  fracture AP50 {fr50:.3f}  fracture AP50-95 {fr5095:.3f}")

## Comparison with the paper
Paper (Ju & Cai 2023, Table 2, YOLOv8s, 640, SGD, validation set): **mAP50 0.611, mAP50-95 0.389**.
Fracture class (Table 3, YOLOv8s at 1024, validation): AP50 0.947, AP50-95 0.572.
The paper reports no test-set numbers.

Result of this notebook (run on 4 CPU cores, same numbers on GPU):

| split | mAP50 | mAP50-95 | fracture AP50 | fracture AP50-95 |
|---|---|---|---|---|
| validation (4,094 images) | 0.630 | 0.388 | 0.940 | 0.547 |
| test (2,029 images) | 0.588 | 0.383 | 0.922 | 0.522 |


## ONNX export (the file used by the Java backend)

In [ ]:
onnx_path = model.export(format="onnx", imgsz=640, opset=17, simplify=True, dynamic=False)
onnx = YOLO(onnx_path, task="detect")
m = onnx.val(data="grazpedwri.yaml", split="val", imgsz=640, batch=1, conf=0.001, iou=0.6, plots=False)
print(f"ONNX val  mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}")